In [ ]:
import pandas as pd
import numpy as np
import os
import re

In [ ]:
# Set benchmark name. Options are recon and flask.
bench_name="recon" #flask

In [ ]:
eval_folder_path = os.path.dirname(os.path.dirname(os.getcwd()))

#Prediction path
path=eval_folder_path+f"/evaluation_results/{bench_name}/"
l=os.listdir(path)
l.sort()

#Labels
gold=pd.read_json(eval_folder_path+f"/benchmarks/{bench_name}/{bench_name}_en.jsonl",lines=True)

In [ ]:
models= ["-".join(file.split("-")[1:]) for file in l]
models = [f for f in pd.Series(models).unique().tolist()]

ioandmono_bench = [m for m in models if re.search(fr"_{bench_name}_(?:io_)?(?:eu|es)\.jsonl$", m)]
io_bench = [m for m in models if re.search(fr"_{bench_name}_io_?(?:eu|es)\.jsonl$", m)]

In [ ]:
# Auxiliary functions

def extract_value(text, lax=True, print_no_match=False):
    """Extracts the score. If lax=True, it will also consider outputs that do not match the prompted format but output a score."""

    pattern = r"\[(RESULT|EMAITZA|RESULTADO)\]\s*([1-5])"
    match = re.search(pattern, text)

    test=re.search(r"\[FORMAT ERROR\]\s*([1-5])", text)
    if match:
        return int(match.group(2).strip())
    elif test and lax:
        return int(test.group(1).strip())
    if print_no_match:
        print(text,"\n\n")
    return np.nan

def get_data(model,lang, addition):
    """mean of predictions
        model: model + bench
        lang: training lang
    """
    
    label= gold["orig_score"].tolist()
    df=pd.DataFrame()
    d1=pd.read_json(path+f"{lang}-"+model,lines=True)
    cols=d1.columns.tolist()[1:]
    d2=pd.read_json(path+f"io_{lang}-"+model,lines=True)
    if addition=="mean":
        df["pred_lang"]=d1[cols].map(extract_value).mean(axis=1)
        df["pred_io_lang"]=d2[cols].map(extract_value).mean(axis=1)
    elif addition=="mode":
        df["pred_lang"]=d1[cols].map(extract_value).mode(axis=1)
        df["pred_io_lang"]=d2[cols].map(extract_value).mode(axis=1)
    df["gold"]=label
    return df

def create_data_in_single_lang_setting (bench_list, lang_list, variant=""):
    """Create dataframe with predictions for models trained in different language settings (mono vs io)
        lang_list: list of training langs
    """
    df=pd.DataFrame()
    for lang in lang_list:
        if lang != "en_es_eu":
            bench_list = [m for m in bench_list if re.search(fr"_recon_{variant}{lang}.jsonl", m)]
        else:
            bench_list=[m for m in bench_list if re.search(fr"_recon_{variant}(eu|es)\.jsonl$", m)]   
        for model in bench_list:
            d1= get_data(model,lang,"mode")
            df=pd.concat([df,d1],axis=0)
    return df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error

def plot_density_sep(df, df_io, lang="lang", save=False):
    """
    df must contain columns:
        gold_lang, gold_io_lang,
        pred_lang_lang, pred_lang_io,
        pred_io_lang_lang, pred_io_lang_io
    """

    df["pred_ev_io"] = df_io["pred_lang"]
    df["pred_io_ev_io"] = df_io["pred_io_lang"]
    
    df = df.dropna()

    sns.set_theme(style="whitegrid", font_scale=1.3)

    cmap_dict = {
        f"{lang} → {lang}": "crest",
        f"{lang} → io_{lang}": "viridis",
        f"io_{lang} → {lang}": "flare",
        f"io_{lang} → io_{lang}": "rocket",
    }

    fig, axes = plt.subplots(2, 2, figsize=(12, 12), sharex=True, sharey=True)

    configs = [
        (0, 0, f"{lang} → {lang}", "gold", "pred_lang"),
        (0, 1, f"{lang} → io_{lang}", "gold", "pred_ev_io"),
        (1, 0, f"io_{lang} → {lang}", "gold", "pred_io_lang"),
        (1, 1, f"io_{lang} → io_{lang}", "gold", "pred_io_ev_io"),
    ]

    for (row, col, title, gold_col, pred_col) in configs:
        ax = axes[row, col]

        # Compute metrics
        r, _ = pearsonr(df[gold_col], df[pred_col])
        mse = mean_squared_error(df[gold_col], df[pred_col])

        # Density plot
        sns.kdeplot(
            data=df, x=gold_col, y=pred_col,
            fill=True, thresh=0.05, levels=60,
            cmap=cmap_dict[title],  # ✅ now matches correctly
            ax=ax,
            alpha=0.9, bw_adjust=0.6
        )

        # Scatter overlay
        sns.scatterplot(
            data=df, x=gold_col, y=pred_col,
            s=20, color="dimgray", alpha=0.3, ax=ax, edgecolor=None
        )

        # Diagonal
        ax.plot([0.5, 5.5], [0.5, 5.5], "--", color="gray", lw=1)

        # Add metrics
        ax.text(
            0.55, 5.4, f"r = {r:.2f}\nMSE = {mse:.2f}",
            fontsize=11, fontweight="bold",
            ha="left", va="top",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7)
        )

        ax.set_xlim(0.5, 5.5)
        ax.set_ylim(0.5, 5.5)
        ax.set_title(title, fontsize=14)
        ax.set_xlabel("Gold rating", fontsize=13)
        ax.set_ylabel("Predicted rating", fontsize=13)

    sns.despine()
    plt.tight_layout()
    if save:
        plt.savefig(f"{bench_name}_density_plot.pdf", bbox_inches="tight", dpi=300)
    plt.show()

In [ ]:
df_mono=create_data_in_single_lang_setting(models,["eu","es","en_es_eu"])
df_eu_mono=create_data_in_single_lang_setting(models,["eu"])
df_es_mono=create_data_in_single_lang_setting(models,["es"])
df_multi_mono=create_data_in_single_lang_setting(models,["en_es_eu"])

In [ ]:
#tested on io benchmark
df_io=create_data_in_single_lang_setting(models,["eu","es","en_es_eu"], variant="io_")
df_eu_io=create_data_in_single_lang_setting(models,["eu"], variant="io_")
df_es_io=create_data_in_single_lang_setting(models,["es"], variant="io_")
df_multi_io=create_data_in_single_lang_setting(models,["en_es_eu"], variant="io_")